# Achilles quickstart: Colab or Kaggle

This notebook sets up [Achilles](https://github.com/Lourdhu02/achilles) in about five minutes. It clones the repo, checks the hardware, runs one lab's reference tests, measures the accelerator and pretrains a small GPT.

**GPU (optional):** go to **Runtime → Change runtime type → T4 GPU**. Every cell also works on a CPU runtime, just more slowly.

> The runtime's disk is wiped when the session ends. Save your `exercise.py` work to a fork or to Google Drive before you close the tab.

In [ ]:
# 1. Hardware check
!nvidia-smi -L || echo "No NVIDIA GPU: running on CPU (fine for every lab's tests)"

In [ ]:
# 2. Clone and install (PyTorch is preinstalled on Colab and Kaggle, so torch is not reinstalled)
import os
if not os.path.exists("achilles"):
    !git clone --depth 1 https://github.com/Lourdhu02/achilles.git
%cd achilles
!pip install -q -r requirements.txt

In [ ]:
# 3. Which backend will the labs use?
import torch
print("torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| compute capability", torch.cuda.get_device_capability(0))
else:
    print("CPU only")

In [ ]:
# 4. Run one lab's reference tests (about a minute)
!python -m pytest labs/05_transformer --impl=solution -q

In [ ]:
# 5. Measure the roofline: matmul TFLOP/s, memory bandwidth, and your hardware tier
!python tools/measure_gpu.py

## Pretrain a small GPT on TinyStories

This downloads the TinyStories validation split (about 20 MB) and trains the reference model from lab 05 for a few hundred steps. On a T4, `train.py` uses fp16 with loss scaling automatically, because the T4 has no fast bf16.

Swap `--impl solution` for `--impl exercise` once your own `labs/05_transformer/exercise.py` passes its tests.

In [ ]:
# 6. Data (the file goes to data/, which git ignores)
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
path = hf_hub_download("roneneldan/TinyStories", "TinyStoriesV2-GPT4-valid.txt", repo_type="dataset", local_dir="data")
print(path)

In [ ]:
# 7. Train: the preset picks model and batch sizes for this hardware
preset = "gpu-8gb" if torch.cuda.is_available() else "cpu"
!python labs/05_transformer/train.py --data data/TinyStoriesV2-GPT4-valid.txt --impl solution --preset {preset} --max-steps 500

## Next: your first lab

```bash
!python -m pytest labs/01_autograd          # 37 failing tests: your first job
!python tools/progress.py                   # your scoreboard across all 17 labs
```

Open `labs/01_autograd/README.md` for the handout and edit `labs/01_autograd/exercise.py` in the file browser on the left. The full learning path is in the [README](https://github.com/Lourdhu02/achilles#readme).